# Dual-basis Combo CNN — L=4, primal vs Hadamard-conjugated basis

Trains the **validated Combo-small CNN** in two simulation bases and compares each run
against its own Lanczos-ED ground truth:

| arm | basis | (h_x, h_z) | exact-at-init symmetry | burden on Block-1 |
|---|---|---|---|---|
| `cnnP` | primal (z) | (0.20, 0.00) | A_v (vertex flips) | none (h_x commutes with A_v) |
| `cnnD` | **dual (x)** | (0.00, 0.20) | **B_p (plaquette flips)** | none (h_z commutes with B_p) |
| `cnnP_m` | primal | (0.10, 0.10) | A_v | dress away the A_v-breaking h_z |
| `cnnD_m` | dual | (0.10, 0.10) | B_p | dress away the B_p-breaking h_x |

The dual basis (`--dual_basis`) Hadamard-conjugates H (σx↔σz: stars → Z-products,
plaquettes → X-products) and swaps the Wilson nonlinearity to **vertex-star products**
with the invariant CNN on the L×L star grid. Conjugation is unitary ⇒ **ED energies are
basis-independent**, so `exact/lanczos_ed.py` runs unchanged and each run compares to the
plain ED file at its own (h_x, h_z).

Expectations: the dual pair `cnnP`/`cnnD` should reach comparable rel-err (paper class
1e-6–1e-7) up to the OBC asymmetry (9 plaquettes vs 16 stars; n_params 1,681 vs 2,633).
At (0.1, 0.1) both arms are approximate — compare whose converged error is lower.
All four points are sign-free in their bases (float64, fast JIT).

## Before you start
* Runtime → **GPU** (any); the ED cell additionally needs a **High-RAM** runtime.
* Have a GitHub token (repo read) and, optionally, a W&B API key ready.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned stack

In [ ]:
%pip install -q "jax[cuda12]==0.5.2" "jaxlib==0.5.1" netket==3.16.1.post1 flax==0.10.4 wandb

In [ ]:
# CRITICAL: stop THIS notebook kernel from preallocating 75% of the GPU.
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # must run BEFORE importing jax

import jax, netket, flax
print("jax", jax.__version__, "| netket", netket.__version__, "| flax", flax.__version__)
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "NO GPU -- fix the runtime type first"

## 3. Mount Google Drive  *(resume point after any runtime restart)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone (or update) the repo into Drive

In [ ]:
import os, getpass, subprocess, shutil

DRIVE_ROOT = "/content/drive/MyDrive/2D-TC-colab"
BRANCH     = "variant3-plaquette-transformer"
REPO_DIR   = os.path.join(DRIVE_ROOT, "2D-TC-dualbasis")   # dedicated dir for THIS experiment
os.makedirs(DRIVE_ROOT, exist_ok=True)

def git(args, secret=None, check=True):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if check and r.returncode != 0:
        clean = (lambda s: s.replace(secret, "***")) if secret else (lambda s: s)
        print("GIT FAILED:", clean(" ".join(args)))
        print(clean(r.stderr).strip())
        raise RuntimeError("git command failed (see the GIT FAILED output above)")
    return r

need_clone = not os.path.isdir(os.path.join(REPO_DIR, ".git"))
if not need_clone and git(["-C", REPO_DIR, "rev-parse", "--is-inside-work-tree"], check=False).returncode != 0:
    print("existing repo looks broken -> removing and re-cloning")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    need_clone = True

tok = getpass.getpass("GitHub token (repo read access): ")
url = f"https://{tok}@github.com/SanzharBissenali/2D-TC.git"
try:
    if need_clone:
        print("cloning", BRANCH, "->", REPO_DIR)
        git(["clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], secret=tok)
        git(["-C", REPO_DIR, "remote", "set-url", "origin",
             "https://github.com/SanzharBissenali/2D-TC.git"])
    else:
        print("repo present; best-effort update of", BRANCH)
        for lk in (".git/index.lock", ".git/shallow.lock", ".git/HEAD.lock"):
            lp = os.path.join(REPO_DIR, lk)
            if os.path.exists(lp):
                os.remove(lp); print("removed stale", lk)
        try:
            git(["-C", REPO_DIR, "fetch", url, "--depth", "1", f"{BRANCH}:refs/remotes/origin/{BRANCH}"], secret=tok)
            git(["-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
        except Exception as e:
            print("update skipped (using existing checkout):", str(e)[:150])
finally:
    del tok, url

os.chdir(REPO_DIR)
head = git(["-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
print("repo:", REPO_DIR, "| branch:", head)
assert "--dual_basis" in open("utils/config.py").read(), "dual-basis code missing -- pull the latest branch"
print("OK: dual-basis code present.")

## 5. Weights & Biases login *(optional — set `HP['WANDB']=False` to skip)*

In [ ]:
import os, getpass, wandb
os.environ["WANDB_MODE"] = "online"
wandb.login(key=getpass.getpass("W&B API key (wandb.ai/authorize): "))
print("W&B login OK")

## 6. Hyperparameters + run points
Validated Combo-small CNN recipe (native TDVP, `dt=0.01`, `diag_shift=6e-5`, 350 steps).

In [ ]:
import os, json

REPO   = os.getcwd()
OUTDIR = os.path.join(REPO, "results", "nqs")
EDDIR  = os.path.join(REPO, "results", "ed")
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(EDDIR, exist_ok=True)

HP = dict(
    LX          = 4,
    N_STEPS     = 350,
    DT          = 0.01,
    DIAG_SHIFT  = 6e-5,
    LR_SCHEDULE = "const",
    N_SAMPLES   = 8192,
    CHUNK_SIZE  = 1024,
    SEED        = 0,
    WANDB       = True,
    WANDB_PROJECT = "2d-tc-transformer",
    WANDB_GROUP   = "dual-basis-L4",
)

# arm -> (hx, hz, dual_basis)
RUNS = {
    "cnnP":   (0.20, 0.00, False),  # primal basis, pure-hx cut (A_v exact, B_p emergent)
    "cnnD":   (0.00, 0.20, True),   # dual basis,   pure-hz cut (B_p exact at init)
    "cnnP_m": (0.10, 0.10, False),  # mixed point, primal (hz breaks A_v)
    "cnnD_m": (0.10, 0.10, True),   # mixed point, dual   (hx breaks B_p)
}

def _tag(hp):
    return f"_dt{hp['DT']:g}_ds{hp['DIAG_SHIFT']:g}" + (f"_s{hp['SEED']}" if hp["SEED"] else "")

def _jobid(arm, hp=None):
    hp = hp or HP
    hx, hz, _ = RUNS[arm]
    return f"L{hp['LX']}_hx{hx:.2f}_hz{hz:.2f}_{arm}{_tag(hp)}"

def _base(arm, hp=None):
    return os.path.join(OUTDIR, f"G-equiv_1_{_jobid(arm, hp)}")

def ed_path(hx, hz):
    return os.path.join(EDDIR, f"ed_L{HP['LX']}_hx{hx:.2f}_hz{hz:.2f}.json")

for arm in RUNS:
    print("jobid:", _jobid(arm))

## 7. ED ground truth — runs FIRST *(needs a High-RAM runtime!)*
`ed_L4_hx0.00_hz0.20.json` ships in the repo; the other two points are computed here
(`--no-observables`, ~10 GB peak each). ED is basis-independent, so no dual flag exists
or is needed on this side.

In [ ]:
import subprocess, psutil

def stream(cmd):
    env = dict(os.environ, PYTHONPATH=REPO, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(["bash", "-lc", cmd], env=env, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()

ram_gb = psutil.virtual_memory().total / 1e9
print(f"system RAM: {ram_gb:.1f} GB")
assert ram_gb > 20, ("Not enough system RAM for L=4 ED (~10 GB peak; standard Colab gets "
                     "OOM-killed). Runtime -> Change runtime type -> High-RAM, then re-run.")

assert os.path.exists(ed_path(0.00, 0.20)), \
    "repo should ship ed_L4_hx0.00_hz0.20.json -- wrong branch?"

for hx, hz in [(0.20, 0.00), (0.10, 0.10)]:
    p = ed_path(hx, hz)
    if os.path.exists(p):
        print("ED present:", p); continue
    rc = stream(f"cd {REPO} && python -u -m exact.lanczos_ed "
                f"--Lx {HP['LX']} --hx {hx} --hz {hz} --k 4 --no-observables --out {p}")
    print("ED exit", rc)
    assert rc == 0, f"ED failed (exit {rc}; -9 = OOM-killed -> use a High-RAM runtime)"

def ed_E0(hx, hz):
    return float(json.load(open(ed_path(hx, hz)))["E0"])

for hx, hz in [(0.20, 0.00), (0.00, 0.20), (0.10, 0.10)]:
    print(f"E0(hx={hx:.2f}, hz={hz:.2f}) = {ed_E0(hx, hz):.8f}")

## 8. Training driver
Watch the startup lines of each run: dual arms must print `dual_basis: True`,
`Custom sampler clusters: plaquette flips (dual basis)`,
`[B_p init-invariance] max |Delta log psi| ~ 1e-15`, and **2633** parameters
(primal arms: **1681**, no B_p gate).

In [ ]:
def run_cnn(arm, force=False, **overrides):
    """Train one arm of RUNS, streaming per-step progress live. Overrides merge onto HP.
    Completion marker: .mpack (written after training; skip-if-complete => resumable).
    Returns the run's base path."""
    hp = {**HP, **overrides}
    hx, hz, dual = RUNS[arm]
    jobid, base = _jobid(arm, hp), _base(arm, hp)
    if not force and os.path.exists(base + ".mpack"):
        print(f"== skip {jobid} (already complete; force=True to re-run) ==")
        return base

    wb = (f"--wandb --wandb_project {hp['WANDB_PROJECT']} --wandb_group {hp['WANDB_GROUP']}"
          if hp["WANDB"] else "")
    dual_flag = "--dual_basis " if dual else ""
    cmd = (
        f"cd {OUTDIR} && python -u {REPO}/main.py "
        f"--outindex 1 --jobid {jobid} "
        f"--Lx {hp['LX']} --hx {hx} --hy 0.0 --hz {hz} {dual_flag}"
        f"--optimizer tdvp --n_steps {hp['N_STEPS']} "
        f"--dt {hp['DT']} --diag_shift {hp['DIAG_SHIFT']} --lr_schedule {hp['LR_SCHEDULE']} "
        f"--symmetric_block cnn "
        f"--architecture Combo --channels_noninv 1,16 --channels_inv 16,8,1 --kernel_size 2 "
        f"--n_samples_fin {hp['N_SAMPLES']} --chunk_size {hp['CHUNK_SIZE']} "
        f"--use_custom_sampler --seed {hp['SEED']} {wb}"
    )
    print(f"RUN {jobid}  ({'DUAL' if dual else 'primal'} basis, tdvp, {hp['N_STEPS']} steps)", flush=True)
    env = dict(os.environ, PYTHONPATH=REPO, PYTHONUNBUFFERED="1",
               XLA_PYTHON_CLIENT_MEM_FRACTION="0.90")
    proc = subprocess.Popen(["bash", "-lc", cmd], env=env, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    print(f"===== {jobid} exit {proc.wait()} =====", flush=True)
    return base

## 9. Train — the dual pair (each exactly symmetric on its own cut)

In [ ]:
run_cnn("cnnP")    # primal, (hx,hz)=(0.20,0.00)

In [ ]:
run_cnn("cnnD")    # dual,   (hx,hz)=(0.00,0.20) -- star-Wilson + conjugated H

## 10. Train — the mixed point (0.10, 0.10)
Both bases are approximate here: the primal net must *learn* the A_v-breaking h_z through
Block-1; the dual net the B_p-breaking h_x. Same ED reference for both.

In [ ]:
run_cnn("cnnP_m")  # primal @ (0.10,0.10)

In [ ]:
run_cnn("cnnD_m")  # dual   @ (0.10,0.10)

## 11. Learning curves, ED comparison, physical ⟨B_p⟩/⟨A_v⟩
`Bp_mean`/`Av_mean` in the run JSON are **physical** (the dual runs measure the
conjugated operators), so the mirror pattern should be explicit:
`cnnP` → ⟨A_v⟩ ≈ 1, ⟨B_p⟩ < 1; `cnnD` → ⟨B_p⟩ ≈ 1, ⟨A_v⟩ < 1.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
hdr = f"{'arm':7s} {'basis':7s} {'(hx,hz)':13s} {'E_final':>14s} {'rel-err':>10s} {'<B_p>':>8s} {'<A_v>':>8s}"
print(hdr); print("-" * len(hdr))
for arm, (hx, hz, dual) in RUNS.items():
    try:
        d = json.load(open(_base(arm) + ".json"))
    except FileNotFoundError:
        print(f"({arm}: no run yet)"); continue
    E = np.array([complex(x).real for x in d["energy"]])
    V = np.array([abs(complex(x)) for x in d["Vscore"]])
    E0 = ed_E0(hx, hz)
    rel = np.abs(E - E0) / abs(E0)
    op = d["order_params"]
    bp = op["Bp_mean"][-1] if op["Bp_mean"] else float("nan")
    av = op["Av_mean"][-1] if op["Av_mean"] else float("nan")
    print(f"{arm:7s} {'dual' if dual else 'primal':7s} ({hx:.2f},{hz:.2f})  "
          f"{E[-1]:14.8f} {rel[-1]:10.2e} {bp:8.4f} {av:8.4f}"
          + ("   <= 1e-5 TARGET MET" if rel[-1] <= 1e-5 else ""))
    ax[0].semilogy(rel, label=f"{arm} ({rel[-1]:.1e})")
    ax[1].semilogy(V, label=arm)
ax[0].set(xlabel="step", ylabel="|E - E0| / |E0|",
          title=f"L={HP['LX']} relative energy error vs own ED")
ax[0].legend()
ax[1].set(xlabel="step", ylabel="V-score", title="V-score (lower = better)")
ax[1].legend()
plt.tight_layout(); plt.show()